## 1. Question

**Can an LLM explain song recommendations using catalog evidence while acknowledging when the connection is weak?**

The previous notebook represented three liked songs as embeddings (lists of numbers that capture features of their text). It averaged those embeddings into a **centroid** a single vector used as a rough profile of the listener's interests and used that vector to rank unheard songs.

This notebook keeps that recommendation step and adds a large language model (LLM) to explain why each result may be relevant. The explanation is limited to connections found in the supplied descriptions and lyrics. It cannot prove that the listener will enjoy a song, and it does not reveal why the embedding model produced a particular score.

## 2. What Is RAG?

Retrieval-Augmented Generation, or **RAG**, combines search with text generation:

1. **Retrieve** relevant information from an external collection.
2. **Augment** the model's prompt with that information and its source identifiers.
3. **Generate** an answer from the supplied evidence.

Here, the external collection is our small song catalog. We should not expect the LLM to remember the exact historical lyric editions chosen for this experiment. Instead, the notebook searches the catalog and places the relevant records directly into the prompt. This is the **augmentation** step. The model receives this evidence only while answering; it is not retrained and its weights do not change.

It is important to separate the two jobs in this example:

1. **Recommend:** MiniLM creates embeddings, and Chroma uses the average embedding of the liked songs to find the three most semantically similar songs that the listener has not already listened to.
2. **Explain:** The notebook retrieves relevant descriptions and lyric passages for the liked and recommended songs and gives that evidence to Llama 3.2 to generate an explanation.

Llama explains the songs that have already been recommended; it does not change their order or choose different songs. Keeping these steps separate helps us identify whether a problem comes from selecting the recommendations or explaining them.

## 3. Hypothesis

**Supplying descriptions and retrieved lyric passages should produce more specific and verifiable explanations than supplying titles and genres alone.** In other words, an explanation backed by evidence should tell us *what* the songs have in common and *where* that claim came from. When the evidence shows little similarity, the model should say that the connection is weak instead of inventing a convincing-sounding reason.

Each retrieved passage has a unique **chunk ID**. The model can cite this ID so we can trace a claim back to the exact description or lyric passage used as evidence.

We will evaluate each explanation using four questions:

- **Specificity:** Does it describe a clear connection between the songs instead of giving generic praise?
- **Valid citations:** Does it refer only to song IDs and chunk IDs included in the prompt?
- **Support:** Are its claims backed by the supplied descriptions and lyrics?
- **Honesty about uncertainty:** Does it acknowledge when the evidence is weak or conflicting?

## 4. Experiment Setup

This experiment uses a small catalog of eight historical songs with public-domain lyrics. A small dataset keeps the full process easy to inspect by hand. It is separate from the 24-song fictional catalog used in the earlier notebooks. We use the same recommendation method as before, averaging the liked songs’ embeddings to find similar songs. We then provide descriptions and lyric passages to help the model explain each recommendation.

Each component has one job:

| Component | Role |
| --- | --- |
| `public-domain-songs.csv` | Song metadata, descriptions, source links, and rights notes |
| `public-domain-song-lyrics.csv` | Source-linked lyric sections |
| `all-MiniLM-L6-v2` | Embed songs and evidence for semantic retrieval |
| Chroma | Store embeddings and retrieve candidates and chunks |
| Listener centroid | Represent the three liked songs as one query vector |
| `llama3.2:3b` | Generate explanations from the retrieved evidence |

A useful shorthand is: **MiniLM retrieves; Llama explains.** For recommendation, each song gets one embedding made from its title and description. For explanation, descriptions and lyrics are stored as smaller, source-linked passages called **chunks**. Keeping the two indexes separate means a song with many lyric sections still counts as one song during recommendation rather than receiving extra influence simply because it has more text.

The dataset contains lyric text, not audio. It does not include modern recordings or arrangements, which can have rights separate from the underlying composition. The short descriptions were written for this experiment; the linked historical sources provide the lyric provenance. Punctuation and spelling are lightly normalized for consistent text processing.

This notebook uses Ollama to run Llama 3.2 locally. If Ollama is not installed, follow the [official Ollama quickstart](https://docs.ollama.com/quickstart). After installation, open Ollama and download the model:

```sh
ollama pull llama3.2:3b
```

Keep Ollama running while you execute the notebook.

The next cell imports the libraries used throughout the notebook and defines three settings: the Ollama model, the generation temperature, and the number of recommendations to return.

**Temperature** controls how much variation the model may introduce. Lower values suit focused tasks, such as explaining a recommendation from supplied evidence; higher values suit creative tasks, such as brainstorming names for a music app. We set it to `0` so differences between the two experiment conditions are easier to attribute to the supplied evidence rather than generation variability. This reduces variation but does not guarantee identical or correct answers.

In [6]:
import csv
import re
from dataclasses import dataclass, field
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import Client

GENERATION_MODEL = "llama3.2:3b"
TEMPERATURE = 0
N_RECOMMENDATIONS = 3

### Load the catalog

The next cell turns rows from the two CSV files into Python objects. A `Song` holds one catalog row. A `LyricSection` holds one verse, chorus, or other section.

They are connected by `song_id` in a **one-to-many relationship**: one song can have many lyric sections, while each lyric section belongs to one song. If database terms are familiar, `Song.song_id` is the primary key and `LyricSection.song_id` is the foreign key.

```text
pd_001: Song
  ├── pd_001: verse_1
  ├── pd_001: verse_2
  └── pd_001: chorus
```

The code also checks that every lyric section points to a song that exists. The path fallback lets the notebook run from either the repository root or the `embeddings-and-search` directory.

In [7]:
@dataclass(frozen=True)
class Song:
    song_id: str
    title: str
    artist: str
    genre: str
    description: str
    source_url: str
    rights_note: str

    def recommendation_text(self) -> str:
        return f"{self.title} {self.description}"


@dataclass(frozen=True)
class LyricSection:
    song_id: str
    section: str
    text: str


catalog_path = Path("public-domain-songs.csv")
lyrics_path = Path("public-domain-song-lyrics.csv")
if not catalog_path.exists():
    catalog_path = Path("embeddings-and-search") / catalog_path
    lyrics_path = Path("embeddings-and-search") / lyrics_path

with catalog_path.open(newline="", encoding="utf-8") as song_file:
    songs = [
        Song(
            song_id=row["id"],
            title=row["title"],
            artist=row["artist"],
            genre=row["genre"],
            description=row["description"],
            source_url=row["source_url"],
            rights_note=row["rights_note"],
        )
        for row in csv.DictReader(song_file)
    ]


with lyrics_path.open(newline="", encoding="utf-8") as lyrics_file:
    lyric_sections = [
        LyricSection(
            song_id=row["song_id"],
            section=row["section"],
            text=row["text"],
        )
        for row in csv.DictReader(lyrics_file)
    ]

songs_by_id = {song.song_id: song for song in songs}
unknown_lyric_ids = {section.song_id for section in lyric_sections} - songs_by_id.keys()
if unknown_lyric_ids:
    raise ValueError(f"Lyrics reference unknown songs: {sorted(unknown_lyric_ids)}")

print(f"Loaded {len(songs)} public-domain songs")
print(f"Loaded {len(lyric_sections)} lyric sections")

Loaded 8 public-domain songs
Loaded 27 lyric sections


### Chunk the explanation evidence

A **chunk** is a smaller passage cut from a larger document. RAG systems usually search chunks instead of entire long documents because a short passage is more likely to focus on one idea. Smaller passages also reduce irrelevant prompt text, use fewer tokens, support precise citations, and make claims easier to check. Very short records should remain intact because splitting them again can remove useful context.

`RecursiveCharacterTextSplitter` divides long text into chunks for retrieval. It first tries to split at natural boundaries such as paragraphs, lines, and sentences and uses smaller boundaries only when a passage is still too long. This helps keep related text together while keeping each chunk near the chosen size. It is useful here because short lyric sections remain intact, while longer sections are divided into focused passages. The search system can then find the most relevant passages and include only those passages in the LLM prompt.

The two indexes embed text for different purposes. The recommendation index uses exactly one title and description document per song, so every song appears once during candidate retrieval. The evidence index separately embeds descriptions and lyric chunks, linked back to their songs by `song_id`, so relevant passages can be retrieved after a recommendation has been selected.

Each song description is stored as one source passage. Lyrics are first divided by their natural structure, such as verses and choruses. If one of these sections is too long, it is split into smaller chunks. Every chunk keeps information about where it came from, including the song, lyric section, source type, source URL, and rights note.

Here, `chunk_size=180` asks the character-based splitter to keep chunks at about 180 characters or fewer. `chunk_overlap=30` repeats about 30 characters from one chunk in the next, helping preserve a phrase that crosses a boundary.

This creates a trade-off:

- Larger chunks preserve more context but may mix unrelated ideas.
- Smaller chunks are more focused but can split an important idea.
- More overlap protects ideas at boundaries but duplicates text, storage, and prompt tokens.

The extra cost is tiny for eight songs, but it matters at larger scale. The values here are teaching defaults, not universal best settings; a production system should test several values with realistic questions.

In [8]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". " , " "],
    chunk_size=180,
    chunk_overlap=30,
)

description_sources = [
    {
        "song": song,
        "source_type": "description",
        "section": "catalog_description",
        "text": song.description,
    }
    for song in songs
]

lyric_sources = [
    {
        "song": songs_by_id[lyric.song_id],
        "source_type": "lyrics",
        "section": lyric.section,
        "text": lyric.text,
    }
    for lyric in lyric_sections
]

evidence_sources = description_sources + lyric_sources

evidence_chunks = []
for source in evidence_sources:
    song = source["song"]
    for chunk_index, text in enumerate(splitter.split_text(source["text"])):
        evidence_chunks.append(
            {
                "id": (
                    f"{song.song_id}:{source['source_type']}:"
                    f"{source['section']}:chunk-{chunk_index}"
                ),
                "text": text,
                "metadata": {
                    "song_id": song.song_id,
                    "title": song.title,
                    "genre": song.genre,
                    "source_type": source["source_type"],
                    "section": source["section"],
                    "source_url": song.source_url,
                    "rights_note": song.rights_note,
                    "chunk_index": chunk_index,
                },
            }
        )

print(f"Created {len(evidence_chunks)} evidence chunks from {len(evidence_sources)} sources")
for chunk in evidence_chunks[:3]:
    print(f"\n{chunk['id']}: {chunk['text']}")

Created 65 evidence chunks from 35 sources

pd_001:description:catalog_description:chunk-0: A song of longing for an absent loved one, remembered through images of summer, flowing water, and a voice that is gone.

pd_002:description:catalog_description:chunk-0: A declaration of lasting devotion to Annie Laurie, whose remembered promise inspires complete loyalty.

pd_003:description:catalog_description:chunk-0: A promise that love will endure after youth and beauty fade, grounded in constancy rather than appearance.


### Build two indexes

The notebook now builds two searchable Chroma collections:

| Collection | Stores | Used for |
| --- | --- | --- |
| Recommendation | One title and description document per song | Finding unheard songs similar to the listener profile |
| Evidence | One document per description or lyric chunk | Finding passages that can support an explanation |

Both collections use `upsert`, which means “insert this ID if it is new, or update it if it already exists.” This makes the cell safe to rerun without creating duplicate records.

In [9]:
client = chromadb.Client()
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

recommendation_collection = client.get_or_create_collection(
    name="public_domain_song_recommendations",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)
recommendation_collection.upsert(
    ids=[song.song_id for song in songs],
    documents=[song.recommendation_text() for song in songs],
)

evidence_collection = client.get_or_create_collection(
    name="public_domain_song_evidence",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)
evidence_collection.upsert(
    ids=[chunk["id"] for chunk in evidence_chunks],
    documents=[chunk["text"] for chunk in evidence_chunks],
    metadatas=[chunk["metadata"] for chunk in evidence_chunks],
)

print(f"Indexed {recommendation_collection.count()} songs")
print(f"Indexed {evidence_collection.count()} evidence chunks")

Indexed 8 songs
Indexed 65 evidence chunks


## 5. Retrieval: Find Candidate Songs

The next two cells create the actual recommendations. The process is:

1. Retrieve the embedding for each liked song.
2. Average those embeddings to create one vector representing the listener's interests.
3. Ask Chroma to find the songs with the most similar embeddings.
4. Exclude songs the listener has already listened to.

The result is a ranked list of recommended songs. The LLM is not involved in this step, so it cannot influence which songs are selected.

Averaging provides a simple representation of the listener's interests, but it works best when the liked songs have related textual themes. If the songs cover very different themes, the average may fall between those interests and represent none of them strongly. This can lead to weaker recommendations. Different genres are not necessarily a problem if their descriptions express similar themes.

In [10]:
@dataclass
class ListeningHistory:
    songs: list[Song] = field(default_factory=list)

    def add(self, song: Song) -> None:
        self.songs.append(song)


def recommend(listening_history: ListeningHistory, n_results: int = 3) -> list[Song]:
    liked_ids = list(dict.fromkeys(song.song_id for song in listening_history.songs))
    if not liked_ids or n_results <= 0:
        return []

    stored = recommendation_collection.get(ids=liked_ids, include=["embeddings"])
    embeddings = stored["embeddings"]
    if embeddings is None or len(embeddings) != len(liked_ids):
        raise ValueError("A liked song is missing an embedding")

    values_by_dimension = zip(*embeddings)

    # Average each dimension across all liked-song embeddings.
    average_embeddings = [
        sum(float(value) for value in dimension_values) / len(embeddings)
        for dimension_values in values_by_dimension
    ]
    results = recommendation_collection.query(
        query_embeddings=[average_embeddings],
        n_results=recommendation_collection.count(),
    )

    recommendations = [
        songs_by_id[song_id]
        for song_id in results["ids"][0]
        if song_id not in liked_ids
    ]
    return recommendations[:n_results]

In [11]:
listener = ListeningHistory()
for song_id in ("pd_001", "pd_002", "pd_003"):
    listener.add(songs_by_id[song_id])

liked_songs = list(listener.songs)
recommendations = recommend(listener, N_RECOMMENDATIONS)

print("Liked songs:")
for song in liked_songs:
    print(f"- {song.song_id}: {song.title} [{song.genre}]\n  {song.description}")
    print(f"  Source: {song.source_url}")

print("\nRetrieved recommendations:")
for song in recommendations:
    print(f"- {song.song_id}: {song.title} [{song.genre}]\n  {song.description}")
    print(f"  Source: {song.source_url}")

Liked songs:
- pd_001: Jeanie with the Light Brown Hair [Parlor song]
  A song of longing for an absent loved one, remembered through images of summer, flowing water, and a voice that is gone.
  Source: https://en.wikisource.org/wiki/Jeanie_with_the_Light_Brown_Hair
- pd_002: Annie Laurie [Scottish folk song]
  A declaration of lasting devotion to Annie Laurie, whose remembered promise inspires complete loyalty.
  Source: https://en.wikisource.org/wiki/Annie_Laurie
- pd_003: Believe Me If All Those Endearing Young Charms [Irish melody]
  A promise that love will endure after youth and beauty fade, grounded in constancy rather than appearance.
  Source: https://en.wikisource.org/wiki/The_Canadian_Soldiers%27_Song_Book/Believe_Me_If_All_Those_Endearing_Young_Charms

Retrieved recommendations:
- pd_006: The Last Rose of Summer [Irish melody]
  A meditation on loneliness and mortality through the image of the final rose left after its companions have faded.
  Source: https://en.wikisource.

## 6. Augmentation: Prepare the Evidence

After selecting a recommendation, the notebook gathers evidence that can help explain why it may relate to the liked songs.

For each recommendation:

1. Combine its description with the descriptions of the liked songs to create a search query.
2. Search for relevant description and lyric chunks.
3. Use a **metadata filter** to limit the search to the liked songs and the recommendation being explained.
4. Keep each retrieved chunk’s ID so the generated explanation can cite its source.

The notebook then prints the exact evidence that will be included in the LLM prompt. This makes it possible to inspect the source material before evaluating the generated explanation.

In [12]:
def retrieve_evidence(
    liked: list[Song],
    recommended: Song,
    chunks_per_song: int = 2,
) -> list[dict]:
    liked_descriptions = " ".join(song.description for song in liked)
    question = (
        f"Recommended song: {recommended.title}. {recommended.description} "
        f"Liked-song themes: {liked_descriptions}"
    )
    records = []
    for song in [*liked, recommended]:
        result = evidence_collection.query(
            query_texts=[question],
            n_results=chunks_per_song,
            where={"song_id": song.song_id},
            include=["documents", "metadatas"],
        )
        for chunk_id, text, metadata in zip(
            result["ids"][0],
            result["documents"][0],
            result["metadatas"][0],
        ):
            records.append(
                {
                    "chunk_id": chunk_id,
                    "text": text,
                    "metadata": metadata,
                    "role": (
                        "recommended"
                        if song.song_id == recommended.song_id
                        else "liked"
                    ),
                }
            )
    return records


def build_metadata_context(liked: list[Song], recommended: Song) -> str:
    songs_for_context = [*liked, recommended]
    return "\n".join(
        f"<song role='{('recommended' if song.song_id == recommended.song_id else 'liked')}' "
        f"id='{song.song_id}' title='{song.title}' genre='{song.genre}' />"
        for song in songs_for_context
    )


def build_context(
    liked: list[Song],
    recommended: Song,
    evidence: list[dict] | None = None,
) -> str:
    metadata_context = build_metadata_context(liked, recommended)
    if evidence is None:
        return metadata_context

    evidence_context = "\n\n".join(
        (
            f"<evidence role='{record['role']}' chunk_id='{record['chunk_id']}' "
            f"song_id='{record['metadata']['song_id']}' "
            f"source_type='{record['metadata']['source_type']}' "
            f"section='{record['metadata']['section']}' "
            f"source_url='{record['metadata']['source_url']}'>\n"
            f"{record['text']}\n"
            f"</evidence>"
        )
        for record in evidence
    )
    return f"{metadata_context}\n\n{evidence_context}"


CITATION_PATTERN = re.compile(r"\bpd_\d{3}(?::[A-Za-z0-9_-]+){0,3}\b")


def check_citations(
    explanation: str,
    liked: list[Song],
    recommended: Song,
    evidence: list[dict] | None = None,
) -> dict:
    evidence = evidence or []
    permitted_ids = {song.song_id for song in [*liked, recommended]}
    permitted_ids.update(record["chunk_id"] for record in evidence)
    cited_ids = set(CITATION_PATTERN.findall(explanation))
    cited_chunk_ids = {citation for citation in cited_ids if ":" in citation}
    return {
        "invalid_ids": sorted(cited_ids - permitted_ids),
        "has_chunk_citation": bool(cited_chunk_ids),
    }


def build_prompt(context: str, recommended: Song) -> str:
    return f"""
Explain why {recommended.title} ({recommended.song_id}) was recommended.

Use only the records in <context>.
- Write no more than three sentences.
- Cite supporting song IDs.
- Every claim based on a description or lyric must cite at least one exact chunk ID from the context.
- Do not invent audio qualities, listener preferences, or catalog facts.
- If the connection is weak or contradictory, say so explicitly.
- Do not promise that the listener will enjoy the song.

<context>
{context}
</context>
""".strip()

In [13]:
example_evidence = retrieve_evidence(liked_songs, recommendations[0])
example_context = build_context(
    liked_songs, recommendations[0], evidence=example_evidence
)
example_prompt = build_prompt(example_context, recommendations[0])
print(example_prompt)

Explain why The Last Rose of Summer (pd_006) was recommended.

Use only the records in <context>.
- Write no more than three sentences.
- Cite supporting song IDs.
- Every claim based on a description or lyric must cite at least one exact chunk ID from the context.
- Do not invent audio qualities, listener preferences, or catalog facts.
- If the connection is weak or contradictory, say so explicitly.
- Do not promise that the listener will enjoy the song.

<context>
<song role='liked' id='pd_001' title='Jeanie with the Light Brown Hair' genre='Parlor song' />
<song role='liked' id='pd_002' title='Annie Laurie' genre='Scottish folk song' />
<song role='liked' id='pd_003' title='Believe Me If All Those Endearing Young Charms' genre='Irish melody' />
<song role='recommended' id='pd_006' title='The Last Rose of Summer' genre='Irish melody' />

<evidence role='liked' chunk_id='pd_001:description:catalog_description:chunk-0' song_id='pd_001' source_type='description' section='catalog_descrip

## 7. Generation: Explain the Recommendations

Llama 3.2 reads the prompt and writes the explanation. The prompt contains three parts:

- a **system message** defining the model's role and evidence rules;
- the liked songs and the recommendation to explain;
- optionally, the retrieved description and lyric chunks.

The system message tells the model to cite supplied IDs, avoid unsupported musical details, and admit when the match is weak. We set **temperature** to zero to reduce randomness and make comparisons easier. Even so, local model output is not guaranteed to be identical across every runtime or hardware setup.

In [14]:
ollama_client = Client()

SYSTEM_MESSAGE = (
    "You explain song recommendations using only supplied catalog evidence. "
    "Treat source identifiers as citations, not as facts by themselves."
)

def generate_explanation(prompt: str) -> str:
    response = ollama_client.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": TEMPERATURE},
    )
    return response.message.content

## 8. Results

We compare two conditions while keeping the listener, recommendations, instructions, model, and temperature fixed:

- **Metadata only:** song IDs, titles, and genres.
- **Retrieved evidence:** the identical metadata records plus source-linked description and lyric chunks.

This is a controlled comparison: only the evidence changes. If the retrieved-evidence explanation is more concrete and checkable, that supports the hypothesis that RAG helps ground the response.

For each evidence-backed result, the notebook prints the available chunk IDs and reports any ID cited by the answer that was absent from the prompt. Passing this check means the citation was available to the model; it does **not** prove that the sentence attached to that citation is accurate. That still requires reading the claim and its source together.

In [15]:
comparison_results = []
for recommended in recommendations:
    for condition, include_evidence in (
        ("metadata_only", False),
        ("retrieved_evidence", True),
    ):
        evidence = (
            retrieve_evidence(liked_songs, recommended)
            if include_evidence
            else None
        )
        context = build_context(liked_songs, recommended, evidence=evidence)
        explanation = generate_explanation(build_prompt(context, recommended))
        citation_check = check_citations(
            explanation, liked_songs, recommended, evidence
        )
        comparison_results.append(
            {
                "song_id": recommended.song_id,
                "title": recommended.title,
                "condition": condition,
                "explanation": explanation,
                "evidence_ids": [
                    record["chunk_id"] for record in (evidence or [])
                ],
                "citation_check": citation_check,
            }
        )

for result in comparison_results:
    print(f"\n{result['title']} — {result['condition']}")
    if result["evidence_ids"]:
        print("Retrieved evidence:")
        for chunk_id in result["evidence_ids"]:
            print(f"- {chunk_id}")
    print(result["explanation"])
    check = result["citation_check"]
    print(f"Invalid citations: {check['invalid_ids'] or 'none'}")
    if result["evidence_ids"]:
        print(f"Chunk citation supplied: {check['has_chunk_citation']}")


The Last Rose of Summer — metadata_only
The Last Rose of Summer was recommended because it shares a similar genre with the already liked songs, specifically Irish melody (pd_003). The recommended song is also in the same genre as Annie Laurie (pd_002), which is a Scottish folk song. This similarity in genre may suggest a connection between the recommended song and the already liked songs.
Invalid citations: none

The Last Rose of Summer — retrieved_evidence
Retrieved evidence:
- pd_001:description:catalog_description:chunk-0
- pd_001:lyrics:verse_2:chunk-1
- pd_002:description:catalog_description:chunk-0
- pd_002:lyrics:verse_1:chunk-0
- pd_003:lyrics:verse_1:chunk-1
- pd_003:description:catalog_description:chunk-0
- pd_006:description:catalog_description:chunk-0
- pd_006:lyrics:verse_1:chunk-0
The Last Rose of Summer (pd_006) was recommended because it shares a similar theme of longing and mortality with Jeanie with the Light Brown Hair (pd_001), as evidenced by the description of bo

## 9. Analysis and Failure Cases

Generated text can sound trustworthy even when it is unsupported, so evaluate the explanation against the printed prompt rather than judging it by fluency. A useful manual audit asks:

| Check | Question |
| --- | --- |
| Specificity | Does it identify a concrete shared or conflicting theme? |
| Citation validity | Was every cited song ID and chunk ID supplied? |
| Faithfulness | Is every factual claim supported by the records? |
| Calibration | Does it acknowledge visibly weak evidence? |

The next cell creates a deliberately mixed listening history as a **stress test**. Because the catalog is tiny and the recommender must always return three candidates, some returned songs may be weak matches. A good explanation should acknowledge that weakness. Ranking third only means “third closest among the available choices”; it does not automatically mean “a strong recommendation.”

In [16]:
mixed_listener = ListeningHistory()
for song_id in ("pd_005", "pd_008"):
    mixed_listener.add(songs_by_id[song_id])

mixed_likes = list(mixed_listener.songs)
mixed_recommendations = recommend(mixed_listener, N_RECOMMENDATIONS)

print("Mixed-listener recommendations:")
for recommended in mixed_recommendations:
    print(f"- {recommended.song_id}: {recommended.title} — {recommended.description}")

print("\nGrounded explanations:")
for recommended in mixed_recommendations:
    evidence = retrieve_evidence(mixed_likes, recommended)
    context = build_context(mixed_likes, recommended, evidence=evidence)
    explanation = generate_explanation(build_prompt(context, recommended))
    check = check_citations(explanation, mixed_likes, recommended, evidence)
    print(f"\n{recommended.title}")
    print("Retrieved evidence:")
    for record in evidence:
        print(f"- {record['chunk_id']}")
    print(explanation)
    print(f"Invalid citations: {check['invalid_ids'] or 'none'}")
    print(f"Chunk citation supplied: {check['has_chunk_citation']}")

Mixed-listener recommendations:
- pd_004: Beautiful Dreamer — A gentle serenade asking a beloved dreamer to awaken after the noise and cares of the day have passed.
- pd_007: Auld Lang Syne — A song about preserving old friendship and shared memories despite time, distance, and separation.
- pd_001: Jeanie with the Light Brown Hair — A song of longing for an absent loved one, remembered through images of summer, flowing water, and a voice that is gone.

Grounded explanations:

Beautiful Dreamer
Retrieved evidence:
- pd_005:description:catalog_description:chunk-0
- pd_005:lyrics:verse_6:chunk-0
- pd_008:description:catalog_description:chunk-0
- pd_008:lyrics:verse_2:chunk-0
- pd_004:description:catalog_description:chunk-0
- pd_004:lyrics:verse_1:chunk-1
Beautiful Dreamer (pd_004) was recommended because it shares similarities with Flow Gently Sweet Afton (pd_005) and Home, Sweet Home (pd_008) in terms of its genre and lyrical themes. Specifically, all three songs are parlor songs, and B

When reviewing the stress-test output, look especially for:

- invented tempo, instrumentation, or performance details;
- claims that the listener *will* enjoy a song;
- citations that were not present in the prompt;
- confident statements that treat different kinds of longing as identical.

The supplied descriptions and lyric passages—not the confidence or fluency of the prose—determine whether a claim is supported.

## 10. Conclusion and Next Steps

Compare the metadata-only and retrieved-evidence outputs before deciding whether the observations support the hypothesis. Evidence-backed explanations should be more specific and easier to check. However, one small catalog and one generated answer per prompt are a demonstration, not proof that the system is reliable.

The central lesson is that **RAG gives an LLM relevant source material, but it does not guarantee faithful use of that material**. Retrieval quality, prompt design, model behavior, and human evaluation all still matter.

A more systematic next experiment would label claims as supported or unsupported, measure citation validity and weak-match acknowledgement across several listener histories, repeat generations, and compare models. Longer licensed or public-domain song notes or lyrics could then make chunking and passage retrieval a larger part of the evaluation.